# Mercury - Free Tier Fine-Tuning (Echo 1.1)

Covers Echo's two engines: Distil-Whisper-Large-v3 (ASR) and Kokoro-82M (TTS).
Full fine-tune and LoRA/PEFT paths for each - see MODELVERSION.md for what a fine-tune does to the version number (1.1 -> 2.0 on first fine-tune, then 2.0 -> 2.1 -> 2.2 for each one after).

## 0. Setup - install dependencies

In [ ]:
!pip install -q transformers==4.47.1 torch==2.5.1 peft accelerate kokoro==0.7.16 soundfile pydub jiwer huggingface_hub datasets

## 1. Convert any WAV source audio to MP3

Run once against your raw dataset directory before building manifests - keeps disk
quota and any exported archives small. Skip if your data is already MP3.

In [ ]:
import os
from pydub import AudioSegment

DATASET_DIR = "/kaggle/input/your-dataset"  # <-- point at your real dataset
OUT_DIR = "/kaggle/working/audio_mp3"
os.makedirs(OUT_DIR, exist_ok=True)

converted = 0
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.lower().endswith(".wav"):
            src = os.path.join(root, f)
            dst = os.path.join(OUT_DIR, os.path.splitext(f)[0] + ".mp3")
            AudioSegment.from_wav(src).export(dst, format="mp3", bitrate="128k")
            converted += 1
print(f"Converted {converted} wav files to mp3 in {OUT_DIR}")

## 2. Build train/val manifests

NeMo-format manifests (audio_filepath/text/duration per line) work for a HF-transformers
training loop too via a small wrapper - built here once, reused by every model below.

In [ ]:
import json
import soundfile as sf

MANIFEST_TRAIN = "/kaggle/working/train_manifest.json"
MANIFEST_VAL = "/kaggle/working/val_manifest.json"

def build_manifest(audio_text_pairs, out_path):
    with open(out_path, "w") as f:
        for audio_path, text in audio_text_pairs:
            info = sf.info(audio_path)
            f.write(json.dumps({
                "audio_filepath": audio_path,
                "text": text,
                "duration": info.duration,
            }) + "\n")

# TODO: populate from your real (audio_path, transcript) pairs once data is ready.
train_pairs = []
val_pairs = []
build_manifest(train_pairs, MANIFEST_TRAIN)
build_manifest(val_pairs, MANIFEST_VAL)
print(f"train={len(train_pairs)} val={len(val_pairs)} examples (0 until real data is wired in)")

---
## 3. ASR - Distil-Whisper-Large-v3 (LoRA/PEFT path)

Smallest/fastest ASR model - good first target to validate the fine-tuning pipeline works
end-to-end before moving to the bigger Apollo/Thoth tiers.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

MODEL_ID = "distil-whisper/distil-large-v3"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training loop intentionally not run yet - wire up a Seq2SeqTrainer against the
# manifests above once train_pairs/val_pairs are populated with real data.

### 3b. ASR - full fine-tune path (no LoRA)

Same model, no PEFT wrapper - every parameter trains. Needs more VRAM/time than the LoRA
path above; use this only if LoRA underperforms on your dataset.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_ID = "distil-whisper/distil-large-v3"
processor_full = WhisperProcessor.from_pretrained(MODEL_ID)
model_full = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model_full.train()
print(f"full fine-tune: {sum(p.numel() for p in model_full.parameters() if p.requires_grad):,} trainable params")

---
## 4. TTS - Kokoro-82M

Kokoro is small enough that LoRA may be unnecessary - a light full fine-tune on a small
accent/vocabulary dataset is plausible within a single GPU session. Evaluate both.

In [ ]:
from kokoro import KPipeline

pipeline = KPipeline(lang_code="a")
# Kokoro doesn't ship a first-party fine-tune recipe as of this writing - community
# recipes exist (search "kokoro-82m fine-tune lora" on HF/GitHub) but aren't vetted here yet.

---
## 5. Evaluation harness (run after any fine-tune, before promoting a model)

WER/CER on a held-out set, compared against the current production checkpoint - don't promote
a fine-tuned model unless it beats the baseline on this set.

In [ ]:
from jiwer import wer, cer

def evaluate(model_transcribe_fn, val_pairs):
    refs, hyps = [], []
    for audio_path, ref_text in val_pairs:
        hyps.append(model_transcribe_fn(audio_path))
        refs.append(ref_text)
    if not refs:
        return {"wer": None, "cer": None, "n": 0}
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "n": len(refs)}

# evaluate(lambda path: ..., val_pairs)  # wire up the real transcribe fn once training runs